# AuditorAgent — Fine-Tuning Phi-3 Mini with QLoRA

**Author:** Damika | **Date:** March 2026  
**Purpose:** Fine-tune Microsoft Phi-3-mini-4k-instruct to serve as the AuditorAgent in an architectural governance MAS  
**Method:** QLoRA (Quantized Low-Rank Adaptation) via PEFT + TRL SFTTrainer  
**Dataset:** ~815 validated & cleaned JSONL rows from synthetic generation pipeline

---

## Table of Contents

| # | Section | Purpose |
|---|---------|---------|
| 1 | Environment Setup | Install dependencies, configure hardware |
| 2 | Dataset Loading & Formatting | Load cleaned data, convert to chat template format |
| 3 | Chain-Aware Train/Validation Split | Prevent data leakage between audit chains |
| 4 | Model & Tokenizer Setup | Load Phi-3 with 4-bit quantization |
| 5 | QLoRA Configuration | Configure LoRA adapters with justification |
| 6 | Training Configuration | Hyperparameters with rationale |
| 7 | Training Execution | Run SFTTrainer with logging |
| 8 | Evaluation — Structural Metrics | JSON validity, schema compliance, score ranges |
| 9 | Evaluation — Semantic Quality | Output quality assessment |
| 10 | Results Visualization | Training curves, metric comparisons |
| 11 | Model Export | Save adapter and merged model |

---

### Methodology Justification

**Why Phi-3-mini-4k-instruct?**
- 3.8B parameters — small enough for QLoRA on a single consumer GPU (16GB VRAM)
- Strong instruction-following capability from base training
- 4K context window sufficient for our dataset (average input ~2,000 tokens)
- Established in the literature for domain specialisation via PEFT

**Why QLoRA?**
- Reduces trainable parameters from 3.8B to ~2-10M (0.05-0.26% of total)
- 4-bit NormalFloat quantisation keeps full model in ~2GB GPU memory
- Matches the parameter-efficient fine-tuning strategy in the project methodology
- Proven to achieve near-full-fine-tuning quality at fraction of compute cost

**Why SFTTrainer (TRL)?**
- Purpose-built for supervised fine-tuning of instruction-following models
- Native integration with PEFT/LoRA and Hugging Face ecosystem
- Handles chat template formatting automatically
- Built-in gradient checkpointing and mixed-precision support

---
## Section 1 — Environment Setup

Install all required packages. This cell is designed for Google Colab or a local environment with CUDA.

In [1]:
# ═══════════════════════════════════════════════════════════════
# INSTALLATION — Run once, then restart runtime if on Colab
# ═══════════════════════════════════════════════════════════════

!pip install -q torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121
!pip install -q transformers>=4.40.0 peft>=0.10.0 trl>=0.8.0
!pip install -q bitsandbytes>=0.43.0 accelerate>=0.30.0
!pip install -q datasets evaluate scikit-learn
!pip install -q matplotlib seaborn pandas numpy
!pip install -q tensorboard

ERROR: Could not find a version that satisfies the requirement torch (from versions: none)
ERROR: No matching distribution found for torch
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if y

In [ ]:
import os
import json
import time
import hashlib
import warnings
import random
from pathlib import Path
from collections import Counter, defaultdict
from typing import Dict, List, Any, Tuple

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import torch
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
    TrainingArguments,
    TrainerCallback,
)
from peft import (
    LoraConfig,
    get_peft_model,
    prepare_model_for_kbit_training,
    PeftModel,
    TaskType,
)
from trl import SFTTrainer, SFTConfig
from datasets import Dataset

warnings.filterwarnings('ignore')
sns.set_theme(style="whitegrid", font_scale=1.1)
plt.rcParams['figure.dpi'] = 120

# ─── Hardware check ───────────────────────────────────────────
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")
else:
    print("WARNING: No GPU detected. Training will be extremely slow on CPU.")

# ─── Reproducibility ──────────────────────────────────────────
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print(f"Random seed set: {SEED}")

In [ ]:
# ═══════════════════════════════════════════════════════════════
# CONFIGURATION — All paths and hyperparameters in one place
# ═══════════════════════════════════════════════════════════════

# ─── Paths ────────────────────────────────────────────────────
DATASET_PATH = "auditor_dataset_final.jsonl" 
MODEL_NAME = "microsoft/Phi-3-mini-4k-instruct"
OUTPUT_DIR = Path("training_output")
OUTPUT_DIR.mkdir(exist_ok=True)
CHECKPOINT_DIR = OUTPUT_DIR / "checkpoints"
CHECKPOINT_DIR.mkdir(exist_ok=True)
FINAL_MODEL_DIR = OUTPUT_DIR / "auditor_agent_model"
FINAL_MODEL_DIR.mkdir(exist_ok=True)
LOGS_DIR = OUTPUT_DIR / "logs"
LOGS_DIR.mkdir(exist_ok=True)

# ─── Training hyperparameters (justified below) ──────────────
CONFIG = {
    # QLoRA
    "lora_r": 32,               # Rank — 32 balances capacity vs efficiency for structured output
    "lora_alpha": 64,            # Alpha = 2*r is standard practice
    "lora_dropout": 0.05,        # Light dropout to prevent overfitting on 815 rows
    "target_modules": ["q_proj", "k_proj", "v_proj", "o_proj",
                        "gate_proj", "up_proj", "down_proj"],  # All linear layers

    # Quantization
    "bits": 4,                   # 4-bit NormalFloat quantization
    "bnb_4bit_compute_dtype": "bfloat16",
    "bnb_4bit_quant_type": "nf4",
    "use_double_quant": True,    # Nested quantization saves memory

    # Training
    "num_epochs": 3,             # 3 epochs — prevents overfitting on small dataset
    "per_device_batch_size": 1,  # Batch size 1 due to long sequences
    "gradient_accumulation": 8,  # Effective batch size = 8
    "learning_rate": 2e-4,       # Standard QLoRA learning rate
    "lr_scheduler": "cosine",    # Cosine decay with warmup
    "warmup_ratio": 0.05,        # 5% warmup steps
    "weight_decay": 0.01,        # Light regularization
    "max_grad_norm": 1.0,        # Gradient clipping

    # Sequence
    "max_seq_length": 4096,      # Phi-3 context window

    # Validation
    "val_ratio": 0.15,           # 15% for validation
    "eval_steps_ratio": 0.1,     # Evaluate every 10% of training
    "save_steps_ratio": 0.25,    # Save checkpoint every 25%

    # Precision
    "fp16": False,
    "bf16": True,                # BFloat16 for modern GPUs
}

print("Configuration loaded:")
for k, v in CONFIG.items():
    print(f"  {k}: {v}")

---
## Section 2 — Dataset Loading & Formatting

The AuditorAgent receives a structured input payload and must produce a structured JSON audit output.
We format each training example as a **chat conversation** matching Phi-3's chat template:

```
<|system|>
{system prompt}
<|end|>
<|user|>
{input payload as JSON}
<|end|>
<|assistant|>
{target audit output as JSON}
<|end|>
```

This format ensures the fine-tuned model learns to:
1. Accept the same structured input the runtime provides
2. Produce the exact JSON output schema the runtime expects
3. Follow the AuditorAgent's behavioural rules (stable issue IDs, progression language, etc.)

In [ ]:
# ═══════════════════════════════════════════════════════════════
# THE AUDITOR SYSTEM PROMPT — Matches main.py exactly
# ═══════════════════════════════════════════════════════════════
# This is the same system prompt the runtime uses when calling the AuditorAgent.
# The fine-tuned model must learn to follow these rules.

AUDITOR_SYSTEM_PROMPT = '''You are the strict architecture auditor.

Audit the architecture plan against:
- frozen confirmed requirements
- rich requirement notes
- cumulative issue ledger
- revision memory
- prior audit history

Main goal:
- First, verify whether previously reported issues were actually fixed.
- Second, identify the most important remaining weaknesses.
- Third, explain clearly why the score stayed the same, improved, or dropped.

Rules:
- Use stable issue IDs whenever the same issue still exists.
- Mark each issue status as one of: unresolved, resolved, downgraded, new.
- Re-check prior unresolved issues before creating new ones.
- If an earlier issue was fixed, keep the same issue ID and mark it resolved.
- If an earlier issue still exists, keep the same issue ID and explain what is still missing.
- Only create a new issue ID if the problem is materially different from previous issues.
- Score the plan against an absolute rubric, not against any approval threshold.
- Do not try to make the plan pass or fail a gate.
- Be willing to score below 9 if the plan has real weaknesses.
- If the score drops, explain the exact reason for the drop.
- If the score does not improve, explain what blocked improvement.
- Prefer the most important unresolved issues over minor nitpicks.

Return JSON only with:
- thinking_summary
- rubric_scores
- summary
- strengths
- concerns
- blocking_issues
- recommendations
- requirement_conflicts
- issue_updates

rubric_scores must include numeric values from 0 to 10 for:
- requirements_alignment
- architecture_quality
- security
- operability
- internal_consistency

Each requirement_conflicts item must include:
- issue_id, field, current_value, proposed_value, exact_reason, severity

Each issue_updates item must include:
- id, title, severity, status, detail

For each issue_updates.detail:
- State whether the issue was fixed, partially fixed, unchanged, or newly introduced.
- Explain exactly what in the plan caused this judgment.
- If the issue affected the score, explain how.

recommendations should:
- focus on the next highest-impact fixes
- be specific enough for the architect to act on in the next round
- avoid vague advice

summary should:
- briefly explain overall quality
- say whether the round meaningfully improved over the prior round
- mention the main reason the score changed or stayed flat'''

print(f"System prompt length: {len(AUDITOR_SYSTEM_PROMPT)} characters")

In [ ]:
# ═══════════════════════════════════════════════════════════════
# LOAD AND FORMAT DATASET
# ═══════════════════════════════════════════════════════════════

def load_dataset(filepath: str) -> List[Dict]:
    # Load JSONL dataset.
    rows = []
    with open(filepath, 'r', encoding='utf-8') as f:
        for line in f:
            line = line.strip()
            if line:
                try:
                    rows.append(json.loads(line))
                except json.JSONDecodeError:
                    pass
    return rows


def format_input_payload(row: Dict) -> str:
    # Extract and format the input payload that the runtime sends to the AuditorAgent.
    # This mirrors the payload construction in main.py lines 2804-2816.
    payload = row.get('input_payload', {})

    # Build the input exactly as the runtime would
    auditor_input = {
        "round": payload.get('round', 1),
        "frozenrequirementcontract": payload.get('frozenrequirementcontract', {}),
        "requirements": payload.get('requirements', {}),
        "acceptedexceptions": payload.get('acceptedexceptions', {}),
        "issueledger": payload.get('issueledger', {}),
        "revisionmemory": payload.get('revisionmemory', {}),
        "previousaudits": payload.get('previousaudits', []),
        "reasonerreviews": payload.get('reasonerreviews', {}),
        "specialistsubplans": payload.get('specialistsubplans', {}),
        "plan": payload.get('plan', {}),
        "bestaudit": payload.get('bestaudit', {}),
    }

    return json.dumps(auditor_input, indent=1, ensure_ascii=False)


def format_target_output(row: Dict) -> str:
    # Format the target audit output - this is what the model must learn to produce.
    # Uses compact keys matching the dataset schema.
    output = row.get('target_output', {})
    return json.dumps(output, indent=1, ensure_ascii=False)


def build_chat_messages(row: Dict) -> List[Dict[str, str]]:
    # Build the chat message list for a single training example.
    # Format: system -> user (input) -> assistant (output)
    return [
        {"role": "system", "content": AUDITOR_SYSTEM_PROMPT},
        {"role": "user", "content": format_input_payload(row)},
        {"role": "assistant", "content": format_target_output(row)},
    ]


# ─── Load dataset ─────────────────────────────────────────────
raw_data = load_dataset(DATASET_PATH)
print(f"Loaded {len(raw_data)} rows from {DATASET_PATH}")

# ─── Format into chat messages ────────────────────────────────
formatted_data = []
for row in raw_data:
    messages = build_chat_messages(row)
    formatted_data.append({
        "sample_id": row.get("sample_id", ""),
        "messages": messages,
        "_chain_fp": hashlib.sha256(
            json.dumps(row.get('input_payload', {}).get('frozenrequirementcontract', {}),
                       sort_keys=True).encode()
        ).hexdigest()[:16],
    })

print(f"Formatted {len(formatted_data)} training examples")

# ─── Token length analysis ────────────────────────────────────
# Quick estimation of sequence lengths
input_lengths = [len(format_input_payload(r)) for r in raw_data]
output_lengths = [len(format_target_output(r)) for r in raw_data]
total_lengths = [i + o + len(AUDITOR_SYSTEM_PROMPT) for i, o in zip(input_lengths, output_lengths)]

print(f"\nCharacter length statistics:")
print(f"  Input:  mean={np.mean(input_lengths):.0f}, max={max(input_lengths)}")
print(f"  Output: mean={np.mean(output_lengths):.0f}, max={max(output_lengths)}")
print(f"  Total:  mean={np.mean(total_lengths):.0f}, max={max(total_lengths)}")
# Rough token estimate (1 token ≈ 4 chars for JSON)
est_tokens = [t / 3.5 for t in total_lengths]
print(f"  Estimated tokens: mean={np.mean(est_tokens):.0f}, max={max(est_tokens):.0f}")
over_limit = sum(1 for t in est_tokens if t > CONFIG['max_seq_length'])
print(f"  Sequences exceeding {CONFIG['max_seq_length']} tokens: {over_limit}")

---
## Section 3 — Chain-Aware Train/Validation Split

**Critical design decision:** We cannot randomly split rows into train and validation sets because
rows within the same audit chain share the frozen requirement contract. If round 1 of a chain
is in training and round 2 is in validation, the model has effectively "seen" the project during
training — this is **data leakage**.

Instead, we split at the **chain level**: all rounds of a given project go entirely into either
the training set or the validation set. This ensures the validation set contains truly unseen projects.

In [ ]:
# ═══════════════════════════════════════════════════════════════
# CHAIN-AWARE SPLIT
# ═══════════════════════════════════════════════════════════════

def chain_aware_split(data: List[Dict], val_ratio: float = 0.15,
                       seed: int = 42) -> Tuple[List[Dict], List[Dict]]:
    # Split data into train/val at the chain level to prevent data leakage.
    # All rows from the same audit chain go into the same split.
    # Group by chain fingerprint
    chains = defaultdict(list)
    for item in data:
        chains[item['_chain_fp']].append(item)

    # Shuffle chain IDs
    chain_ids = list(chains.keys())
    rng = random.Random(seed)
    rng.shuffle(chain_ids)

    # Split chains
    n_val_chains = max(1, int(len(chain_ids) * val_ratio))
    val_chain_ids = set(chain_ids[:n_val_chains])
    train_chain_ids = set(chain_ids[n_val_chains:])

    train_data = []
    val_data = []
    for cid in train_chain_ids:
        train_data.extend(chains[cid])
    for cid in val_chain_ids:
        val_data.extend(chains[cid])

    return train_data, val_data


train_data, val_data = chain_aware_split(formatted_data, CONFIG['val_ratio'], SEED)

print(f"Chain-aware split:")
print(f"  Total chains: {len(set(d['_chain_fp'] for d in formatted_data))}")
print(f"  Train chains: {len(set(d['_chain_fp'] for d in train_data))}")
print(f"  Val chains:   {len(set(d['_chain_fp'] for d in val_data))}")
print(f"  Train rows: {len(train_data)} ({len(train_data)/len(formatted_data)*100:.1f}%)")
print(f"  Val rows:   {len(val_data)} ({len(val_data)/len(formatted_data)*100:.1f}%)")

# Verify no chain leakage
train_chains = set(d['_chain_fp'] for d in train_data)
val_chains = set(d['_chain_fp'] for d in val_data)
assert len(train_chains & val_chains) == 0, "Chain leakage detected!"
print(f"  ✓ No chain leakage verified")

# Convert to HuggingFace Dataset
train_dataset = Dataset.from_list([{"messages": d["messages"]} for d in train_data])
val_dataset = Dataset.from_list([{"messages": d["messages"]} for d in val_data])

print(f"\nHuggingFace datasets created:")
print(f"  Train: {train_dataset}")
print(f"  Val:   {val_dataset}")

---
## Section 4 — Model & Tokenizer Setup

Load Phi-3-mini-4k-instruct with 4-bit NormalFloat quantisation. The quantisation configuration
reduces the model's memory footprint from ~7.6GB (FP16) to ~2GB, enabling training on consumer GPUs.

**Quantisation choices:**
- `nf4` (NormalFloat4): Optimal for normally-distributed weights (which transformer weights are)
- `double_quant`: Quantises the quantisation constants themselves, saving ~0.4GB
- `bfloat16` compute dtype: Maintains numerical stability during forward/backward passes

In [ ]:
# ═══════════════════════════════════════════════════════════════
# QUANTIZATION CONFIG
# ═══════════════════════════════════════════════════════════════

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type=CONFIG["bnb_4bit_quant_type"],
    bnb_4bit_compute_dtype=getattr(torch, CONFIG["bnb_4bit_compute_dtype"]),
    bnb_4bit_use_double_quant=CONFIG["use_double_quant"],
)

print("BitsAndBytes quantization config:")
print(f"  4-bit quantization: True")
print(f"  Quant type: {CONFIG['bnb_4bit_quant_type']}")
print(f"  Compute dtype: {CONFIG['bnb_4bit_compute_dtype']}")
print(f"  Double quantization: {CONFIG['use_double_quant']}")

# ═══════════════════════════════════════════════════════════════
# LOAD TOKENIZER
# ═══════════════════════════════════════════════════════════════

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME,
    trust_remote_code=True,
    use_fast=True,
)

# Phi-3 uses <|endoftext|> as EOS and <|end|> as chat turn separator
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"  # Required for QLoRA training

print(f"\nTokenizer loaded: {MODEL_NAME}")
print(f"  Vocab size: {tokenizer.vocab_size}")
print(f"  Pad token: {tokenizer.pad_token}")
print(f"  EOS token: {tokenizer.eos_token}")
print(f"  Model max length: {tokenizer.model_max_length}")

# ═══════════════════════════════════════════════════════════════
# LOAD MODEL
# ═══════════════════════════════════════════════════════════════

print(f"\nLoading model: {MODEL_NAME}...")
print("This may take 2-5 minutes on first run (downloading ~2.2GB)...")

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
    attn_implementation="eager",  # Use eager attention for compatibility
    torch_dtype=getattr(torch, CONFIG["bnb_4bit_compute_dtype"]),
)

# Disable cache for training (incompatible with gradient checkpointing)
model.config.use_cache = False
model.config.pretraining_tp = 1

# Prepare for QLoRA training
model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=True)

total_params = sum(p.numel() for p in model.parameters())
print(f"\nModel loaded successfully!")
print(f"  Total parameters: {total_params:,}")
print(f"  Model dtype: {next(model.parameters()).dtype}")
if torch.cuda.is_available():
    print(f"  GPU memory used: {torch.cuda.memory_allocated() / 1e9:.2f} GB")

---
## Section 5 — QLoRA Configuration

**LoRA Hyperparameter Justification:**

| Parameter | Value | Rationale |
|-----------|-------|-----------|
| `r` (rank) | 32 | Higher rank (vs typical 8-16) because our task requires generating complex structured JSON with specific fields, scores, and reasoning. The model needs sufficient adapter capacity to learn the audit schema. |
| `alpha` | 64 | Standard practice: `alpha = 2 * r`. Controls the scaling of LoRA weight updates. |
| `dropout` | 0.05 | Light dropout. Our dataset is small (~815 rows) so some regularisation helps, but too much dropout would prevent the model from learning the precise JSON structure. |
| `target_modules` | All linear layers | We target all attention projections (q, k, v, o) AND MLP layers (gate, up, down). This gives the adapter maximum expressiveness for learning a new output domain (structured audit JSON). |

**Why target all linear layers?**
Standard LoRA often targets only attention layers (q_proj, v_proj). However, our task fundamentally changes
the model's output distribution — it must produce valid JSON with specific keys, numeric scores, and
structured reasoning rather than natural language. Targeting MLP layers (gate_proj, up_proj, down_proj)
allows the model to learn new token-generation patterns needed for this domain shift.

In [ ]:
# ═══════════════════════════════════════════════════════════════
# LoRA CONFIGURATION
# ═══════════════════════════════════════════════════════════════

lora_config = LoraConfig(
    r=CONFIG["lora_r"],
    lora_alpha=CONFIG["lora_alpha"],
    lora_dropout=CONFIG["lora_dropout"],
    target_modules=CONFIG["target_modules"],
    bias="none",                 # Don't train biases — saves parameters
    task_type=TaskType.CAUSAL_LM,
)

# Apply LoRA to the model
model = get_peft_model(model, lora_config)

# Print parameter summary
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
total_params = sum(p.numel() for p in model.parameters())
trainable_pct = trainable_params / total_params * 100

print(f"LoRA Configuration Applied:")
print(f"  Rank (r): {CONFIG['lora_r']}")
print(f"  Alpha: {CONFIG['lora_alpha']}")
print(f"  Dropout: {CONFIG['lora_dropout']}")
print(f"  Target modules: {CONFIG['target_modules']}")
print(f"\nParameter Summary:")
print(f"  Total parameters:     {total_params:>12,}")
print(f"  Trainable parameters: {trainable_params:>12,}")
print(f"  Trainable %:          {trainable_pct:>11.4f}%")
print(f"\n  → Training only {trainable_pct:.2f}% of the model")

# Detailed breakdown
model.print_trainable_parameters()

---
## Section 6 — Training Configuration

**Hyperparameter Rationale:**

| Parameter | Value | Justification |
|-----------|-------|---------------|
| Epochs | 3 | Small dataset (815 rows) overfits quickly. 3 epochs gives ~3 passes without memorisation. Literature shows QLoRA converges in 1-3 epochs for instruction tuning. |
| Batch size | 1 (effective 8) | Long sequences (~2000 tokens) require batch_size=1 per device. Gradient accumulation of 8 steps gives effective batch of 8 for stable gradients. |
| Learning rate | 2e-4 | Standard for QLoRA (Dettmers et al., 2023). Higher than full fine-tuning because only adapter weights are updated. |
| Scheduler | Cosine | Smooth decay prevents sudden LR drops that could destabilise structured output learning. |
| Warmup | 5% | Brief warmup to stabilise early gradients before main training. |
| Weight decay | 0.01 | Light L2 regularisation to prevent overfitting on small dataset. |
| Max grad norm | 1.0 | Gradient clipping prevents exploding gradients common with quantised models. |

In [ ]:
# ═══════════════════════════════════════════════════════════════
# TRAINING ARGUMENTS
# ═══════════════════════════════════════════════════════════════

total_train_steps = len(train_dataset) * CONFIG['num_epochs'] // CONFIG['gradient_accumulation']
eval_steps = max(1, int(total_train_steps * CONFIG['eval_steps_ratio']))
save_steps = max(1, int(total_train_steps * CONFIG['save_steps_ratio']))

print(f"Training plan:")
print(f"  Total training examples: {len(train_dataset)}")
print(f"  Epochs: {CONFIG['num_epochs']}")
print(f"  Batch size (per device): {CONFIG['per_device_batch_size']}")
print(f"  Gradient accumulation: {CONFIG['gradient_accumulation']}")
print(f"  Effective batch size: {CONFIG['per_device_batch_size'] * CONFIG['gradient_accumulation']}")
print(f"  Estimated total steps: {total_train_steps}")
print(f"  Eval every: {eval_steps} steps")
print(f"  Save every: {save_steps} steps")

training_args = SFTConfig(
    output_dir=str(CHECKPOINT_DIR),
    num_train_epochs=CONFIG['num_epochs'],
    per_device_train_batch_size=CONFIG['per_device_batch_size'],
    per_device_eval_batch_size=CONFIG['per_device_batch_size'],
    gradient_accumulation_steps=CONFIG['gradient_accumulation'],
    learning_rate=CONFIG['learning_rate'],
    lr_scheduler_type=CONFIG['lr_scheduler'],
    warmup_ratio=CONFIG['warmup_ratio'],
    weight_decay=CONFIG['weight_decay'],
    max_grad_norm=CONFIG['max_grad_norm'],
    max_seq_length=CONFIG['max_seq_length'],
    fp16=CONFIG['fp16'],
    bf16=CONFIG['bf16'],
    logging_dir=str(LOGS_DIR),
    logging_steps=5,
    eval_strategy="steps",
    eval_steps=eval_steps,
    save_strategy="steps",
    save_steps=save_steps,
    save_total_limit=3,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},
    optim="paged_adamw_8bit",   # Memory-efficient optimizer for QLoRA
    report_to="tensorboard",
    seed=SEED,
    dataset_text_field=None,     # We use messages format, not text field
    packing=False,               # Don't pack — each example is one audit
)

print(f"\nTraining arguments configured.")

---
## Section 7 — Training Execution

We use TRL's `SFTTrainer` which handles:
- Chat template application (automatically formats messages into Phi-3's expected format)
- Loss masking (only computes loss on assistant responses, not system/user tokens)
- Integration with PEFT/LoRA
- Gradient checkpointing for memory efficiency

In [ ]:
# ═══════════════════════════════════════════════════════════════
# CUSTOM CALLBACK — Track training metrics for visualization
# ═══════════════════════════════════════════════════════════════

class MetricsCallback(TrainerCallback):
    # Collect training metrics for post-training visualization.

    def __init__(self):
        self.train_losses = []
        self.eval_losses = []
        self.learning_rates = []
        self.steps = []
        self.eval_steps = []

    def on_log(self, args, state, control, logs=None, **kwargs):
        if logs:
            if 'loss' in logs:
                self.train_losses.append(logs['loss'])
                self.steps.append(state.global_step)
            if 'learning_rate' in logs:
                self.learning_rates.append(logs['learning_rate'])
            if 'eval_loss' in logs:
                self.eval_losses.append(logs['eval_loss'])
                self.eval_steps.append(state.global_step)


metrics_callback = MetricsCallback()

# ═══════════════════════════════════════════════════════════════
# INITIALIZE TRAINER
# ═══════════════════════════════════════════════════════════════

trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    processing_class=tokenizer,
    peft_config=None,  # Already applied via get_peft_model
    callbacks=[metrics_callback],
)

print("SFTTrainer initialized successfully.")
print(f"  Training examples: {len(train_dataset)}")
print(f"  Validation examples: {len(val_dataset)}")

In [ ]:
# ═══════════════════════════════════════════════════════════════
# TRAIN!
# ═══════════════════════════════════════════════════════════════

print("Starting training...")
print(f"{'='*60}")
start_time = time.time()

train_result = trainer.train()

elapsed = time.time() - start_time
print(f"{'='*60}")
print(f"Training complete!")
print(f"  Duration: {elapsed/60:.1f} minutes")
print(f"  Final train loss: {train_result.training_loss:.4f}")
print(f"  Total steps: {train_result.global_step}")

# Save training metrics
metrics = train_result.metrics
metrics['train_runtime_minutes'] = elapsed / 60
trainer.log_metrics("train", metrics)
trainer.save_metrics("train", metrics)

print(f"\nTraining metrics saved to {CHECKPOINT_DIR}")

In [ ]:
# ═══════════════════════════════════════════════════════════════
# EVALUATE ON VALIDATION SET
# ═══════════════════════════════════════════════════════════════

print("Running evaluation on validation set...")
eval_results = trainer.evaluate()

print(f"\nEvaluation Results:")
for key, value in eval_results.items():
    print(f"  {key}: {value:.4f}" if isinstance(value, float) else f"  {key}: {value}")

trainer.log_metrics("eval", eval_results)
trainer.save_metrics("eval", eval_results)

# Compute perplexity from eval loss
eval_loss = eval_results.get('eval_loss', 0)
perplexity = np.exp(eval_loss) if eval_loss < 20 else float('inf')
print(f"\nValidation perplexity: {perplexity:.2f}")
print(f"  (Lower is better. Perplexity < 5 is excellent for domain-specific fine-tuning)")

---
## Section 8 — Evaluation: Structural Metrics

Loss and perplexity measure token-level quality, but we need task-specific metrics to
verify the model produces **structurally valid audit outputs**. We generate outputs on
the validation set and measure:

1. **JSON Validity Rate**: % of outputs that parse as valid JSON
2. **Schema Compliance Rate**: % of valid JSON outputs containing all required keys
3. **Rubric Score Range Compliance**: % of rubric scores within 0-10
4. **Issue Template Compliance**: % of issue_updates following the required detail format
5. **Severity/Status Enum Compliance**: % using only allowed enum values

In [ ]:
# ═══════════════════════════════════════════════════════════════
# GENERATE PREDICTIONS ON VALIDATION SET
# ═══════════════════════════════════════════════════════════════
import re

def generate_audit(model, tokenizer, input_payload: str,
                    max_new_tokens: int = 1500) -> str:
    # Generate an audit output for a given input payload.
    messages = [
        {"role": "system", "content": AUDITOR_SYSTEM_PROMPT},
        {"role": "user", "content": input_payload},
    ]

    # Apply chat template
    prompt = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )

    inputs = tokenizer(prompt, return_tensors="pt", truncation=True,
                        max_length=CONFIG['max_seq_length'] - max_new_tokens)
    inputs = {k: v.to(model.device) for k, v in inputs.items()}

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=0.1,
            do_sample=True,
            top_p=0.9,
            repetition_penalty=1.1,
            pad_token_id=tokenizer.pad_token_id,
        )

    # Decode only the generated tokens (exclude the prompt)
    generated = outputs[0][inputs['input_ids'].shape[1]:]
    return tokenizer.decode(generated, skip_special_tokens=True)


def extract_json_from_text(text: str) -> dict:
    # Try to extract a JSON object from model output text.
    text = text.strip()
    # Try direct parse
    try:
        return json.loads(text)
    except:
        pass
    # Try to find JSON object boundaries
    start = text.find('{')
    end = text.rfind('}')
    if start != -1 and end != -1 and end > start:
        try:
            return json.loads(text[start:end+1])
        except:
            pass
    return None


# ─── Generate on validation set (sample for speed) ───────────
EVAL_SAMPLE_SIZE = min(50, len(val_data))  # Evaluate on up to 50 examples
eval_sample = random.sample(val_data, EVAL_SAMPLE_SIZE)

print(f"Generating predictions on {EVAL_SAMPLE_SIZE} validation examples...")
print("This may take 10-30 minutes depending on GPU speed.")

predictions = []
for i, item in enumerate(eval_sample):
    # Extract the user message (input payload)
    user_msg = item['messages'][1]['content']
    reference = item['messages'][2]['content']  # Ground truth

    if (i + 1) % 10 == 0:
        print(f"  Generating {i+1}/{EVAL_SAMPLE_SIZE}...")

    try:
        generated = generate_audit(model, tokenizer, user_msg)
        parsed = extract_json_from_text(generated)
    except Exception as e:
        generated = ""
        parsed = None

    predictions.append({
        "sample_id": item.get("sample_id", f"val_{i}"),
        "generated_raw": generated,
        "generated_parsed": parsed,
        "reference": json.loads(reference) if reference else None,
    })

print(f"Generation complete. {len(predictions)} predictions.")

In [ ]:
# ═══════════════════════════════════════════════════════════════
# STRUCTURAL METRICS
# ═══════════════════════════════════════════════════════════════

REQUIRED_OUTPUT_KEYS = [
    "thinkingsummary", "rubricscores", "summary", "strengths",
    "concerns", "blockingissues", "recommendations",
    "requirementconflicts", "issueupdates"
]
RUBRIC_DIMS = ["requirementsalignment", "architecturequality",
               "security", "operability", "internalconsistency"]
ALLOWED_SEVERITY = {"critical", "high", "medium", "low"}
ALLOWED_STATUS = {"unresolved", "resolved", "downgraded", "new"}
DETAIL_PATTERN = re.compile(r"^(Fixed|Partially fixed|Unchanged|Newly introduced)\.?",
                             re.IGNORECASE)

# Compute metrics
json_valid = 0
schema_compliant = 0
rubric_valid = 0
issue_template_valid = 0
severity_status_valid = 0
total_issues = 0

for pred in predictions:
    parsed = pred['generated_parsed']

    # 1. JSON validity
    if parsed is not None:
        json_valid += 1
    else:
        continue

    # 2. Schema compliance (all required keys present)
    if all(k in parsed for k in REQUIRED_OUTPUT_KEYS):
        schema_compliant += 1

    # 3. Rubric score range (all 0-10)
    rubric = parsed.get('rubricscores', {})
    if isinstance(rubric, dict):
        scores_ok = all(
            isinstance(rubric.get(d), (int, float)) and 0 <= rubric.get(d, -1) <= 10
            for d in RUBRIC_DIMS if d in rubric
        )
        if scores_ok and len([d for d in RUBRIC_DIMS if d in rubric]) == 5:
            rubric_valid += 1

    # 4 & 5. Issue template and enum compliance
    issues = parsed.get('issueupdates', [])
    if isinstance(issues, list):
        for iu in issues:
            if not isinstance(iu, dict):
                continue
            total_issues += 1

            # Severity/status enums
            sev = str(iu.get('severity', '')).lower()
            stat = str(iu.get('status', '')).lower()
            if sev in ALLOWED_SEVERITY and stat in ALLOWED_STATUS:
                severity_status_valid += 1

            # Detail template
            detail = str(iu.get('detail', ''))
            if DETAIL_PATTERN.match(detail):
                issue_template_valid += 1

n = len(predictions)
n_valid = max(json_valid, 1)

structural_metrics = {
    "json_validity_rate": json_valid / n,
    "schema_compliance_rate": schema_compliant / n,
    "rubric_range_compliance": rubric_valid / n,
    "issue_template_compliance": issue_template_valid / max(total_issues, 1),
    "severity_status_compliance": severity_status_valid / max(total_issues, 1),
}

print(f"\n{'='*60}")
print(f"STRUCTURAL EVALUATION METRICS (n={n})")
print(f"{'='*60}")
for metric, value in structural_metrics.items():
    bar = '█' * int(value * 30)
    print(f"  {metric:35s}: {value:.1%} {bar}")

# Save metrics
with open(OUTPUT_DIR / 'structural_metrics.json', 'w') as f:
    json.dump(structural_metrics, f, indent=2)
print(f"\nMetrics saved to {OUTPUT_DIR / 'structural_metrics.json'}")

---
## Section 9 — Evaluation: Semantic Quality Metrics

Beyond structural validity, we measure how close the model's outputs are to the ground truth:

1. **Rubric Score MAE**: Mean absolute error between predicted and reference rubric scores
2. **Issue Detection F1**: Whether the model identifies the same issues as the reference
3. **Verdict Agreement**: Whether the model reaches similar conclusions (blocking issues, recommendations)

In [ ]:
# ═══════════════════════════════════════════════════════════════
# SEMANTIC EVALUATION METRICS
# ═══════════════════════════════════════════════════════════════

rubric_errors = {d: [] for d in RUBRIC_DIMS}
issue_id_matches = []
blocking_issue_matches = []

for pred in predictions:
    parsed = pred['generated_parsed']
    ref = pred['reference']

    if parsed is None or ref is None:
        continue

    # ── Rubric Score MAE ──
    pred_rubric = parsed.get('rubricscores', {})
    ref_rubric = ref.get('rubricscores', {})

    if isinstance(pred_rubric, dict) and isinstance(ref_rubric, dict):
        for dim in RUBRIC_DIMS:
            pred_score = pred_rubric.get(dim)
            ref_score = ref_rubric.get(dim)
            if isinstance(pred_score, (int, float)) and isinstance(ref_score, (int, float)):
                rubric_errors[dim].append(abs(pred_score - ref_score))

    # ── Issue ID Detection ──
    pred_issues = {str(iu.get('id', '')) for iu in parsed.get('issueupdates', [])
                    if isinstance(iu, dict)}
    ref_issues = {str(iu.get('id', '')) for iu in ref.get('issueupdates', [])
                   if isinstance(iu, dict)}

    if ref_issues:
        # Precision and recall for issue detection
        tp = len(pred_issues & ref_issues)
        precision = tp / max(len(pred_issues), 1)
        recall = tp / max(len(ref_issues), 1)
        f1 = 2 * precision * recall / max(precision + recall, 1e-8)
        issue_id_matches.append({"precision": precision, "recall": recall, "f1": f1})

    # ── Blocking Issue Agreement ──
    pred_blocking = len(parsed.get('blockingissues', []))
    ref_blocking = len(ref.get('blockingissues', []))
    # Agreement: both have blocking issues, or both have none
    agreement = (pred_blocking > 0) == (ref_blocking > 0)
    blocking_issue_matches.append(agreement)


# ── Print Results ──
print(f"\n{'='*60}")
print(f"SEMANTIC EVALUATION METRICS")
print(f"{'='*60}")

print(f"\n── Rubric Score MAE (lower is better) ──")
overall_mae_values = []
for dim in RUBRIC_DIMS:
    if rubric_errors[dim]:
        mae = np.mean(rubric_errors[dim])
        overall_mae_values.extend(rubric_errors[dim])
        print(f"  {dim:30s}: MAE = {mae:.2f}")
    else:
        print(f"  {dim:30s}: No data")

if overall_mae_values:
    print(f"  {'OVERALL':30s}: MAE = {np.mean(overall_mae_values):.2f}")

print(f"\n── Issue Detection ──")
if issue_id_matches:
    avg_precision = np.mean([m['precision'] for m in issue_id_matches])
    avg_recall = np.mean([m['recall'] for m in issue_id_matches])
    avg_f1 = np.mean([m['f1'] for m in issue_id_matches])
    print(f"  Precision: {avg_precision:.2%}")
    print(f"  Recall:    {avg_recall:.2%}")
    print(f"  F1 Score:  {avg_f1:.2%}")
else:
    print(f"  No issue data for comparison")

print(f"\n── Blocking Issue Agreement ──")
if blocking_issue_matches:
    agreement_rate = np.mean(blocking_issue_matches)
    print(f"  Agreement rate: {agreement_rate:.2%}")

# Save all semantic metrics
semantic_metrics = {
    "rubric_mae_overall": float(np.mean(overall_mae_values)) if overall_mae_values else None,
    "rubric_mae_per_dim": {d: float(np.mean(rubric_errors[d])) if rubric_errors[d] else None
                            for d in RUBRIC_DIMS},
    "issue_detection_f1": float(avg_f1) if issue_id_matches else None,
    "issue_detection_precision": float(avg_precision) if issue_id_matches else None,
    "issue_detection_recall": float(avg_recall) if issue_id_matches else None,
    "blocking_agreement": float(agreement_rate) if blocking_issue_matches else None,
}

with open(OUTPUT_DIR / 'semantic_metrics.json', 'w') as f:
    json.dump(semantic_metrics, f, indent=2)
print(f"\nSemantic metrics saved to {OUTPUT_DIR / 'semantic_metrics.json'}")

---
## Section 10 — Results Visualization

Publication-quality charts for the thesis and viva demonstration.

In [ ]:
# ═══════════════════════════════════════════════════════════════
# TRAINING CURVES
# ═══════════════════════════════════════════════════════════════

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# 1. Training loss curve
if metrics_callback.train_losses:
    axes[0, 0].plot(metrics_callback.steps, metrics_callback.train_losses,
                     color='#4C72B0', alpha=0.7, linewidth=0.8, label='Train loss')
    # Add smoothed curve
    if len(metrics_callback.train_losses) > 10:
        window = min(20, len(metrics_callback.train_losses) // 3)
        smoothed = pd.Series(metrics_callback.train_losses).rolling(window, center=True).mean()
        axes[0, 0].plot(metrics_callback.steps, smoothed, color='#C44E52',
                         linewidth=2, label=f'Smoothed (window={window})')
    axes[0, 0].set_xlabel('Step')
    axes[0, 0].set_ylabel('Loss')
    axes[0, 0].set_title('Training Loss')
    axes[0, 0].legend()

# 2. Validation loss curve
if metrics_callback.eval_losses:
    axes[0, 1].plot(metrics_callback.eval_steps, metrics_callback.eval_losses,
                     'o-', color='#55A868', linewidth=2, markersize=6)
    axes[0, 1].set_xlabel('Step')
    axes[0, 1].set_ylabel('Loss')
    axes[0, 1].set_title('Validation Loss')

# 3. Learning rate schedule
if metrics_callback.learning_rates:
    lr_steps = metrics_callback.steps[:len(metrics_callback.learning_rates)]
    axes[1, 0].plot(lr_steps, metrics_callback.learning_rates,
                     color='#DD8452', linewidth=2)
    axes[1, 0].set_xlabel('Step')
    axes[1, 0].set_ylabel('Learning Rate')
    axes[1, 0].set_title('Learning Rate Schedule (Cosine)')

# 4. Structural metrics bar chart
if structural_metrics:
    metric_names = list(structural_metrics.keys())
    metric_values = [structural_metrics[m] for m in metric_names]
    short_names = ['JSON Valid', 'Schema', 'Rubric Range',
                    'Issue Template', 'Sev/Status Enum']
    bars = axes[1, 1].bar(short_names, metric_values, color='#4C72B0',
                           edgecolor='black', alpha=0.8)
    axes[1, 1].set_ylim(0, 1.05)
    axes[1, 1].set_ylabel('Compliance Rate')
    axes[1, 1].set_title('Structural Output Quality')
    axes[1, 1].tick_params(axis='x', rotation=30)
    # Add value labels on bars
    for bar, val in zip(bars, metric_values):
        axes[1, 1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02,
                         f'{val:.1%}', ha='center', fontsize=10)

plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'training_results.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved: training_results.png")

In [ ]:
# ═══════════════════════════════════════════════════════════════
# SEMANTIC EVALUATION VISUALIZATION
# ═══════════════════════════════════════════════════════════════

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# 1. Rubric Score MAE per dimension
if overall_mae_values:
    dim_names = [d.replace('requirements', 'req').replace('architecture', 'arch')
                  .replace('internal', 'int').replace('consistency', 'consist')
                  for d in RUBRIC_DIMS]
    dim_maes = [np.mean(rubric_errors[d]) if rubric_errors[d] else 0 for d in RUBRIC_DIMS]
    colors = ['#55A868' if m < 1.5 else '#DD8452' if m < 2.5 else '#C44E52' for m in dim_maes]

    bars = axes[0].bar(dim_names, dim_maes, color=colors, edgecolor='black', alpha=0.8)
    axes[0].set_ylabel('Mean Absolute Error')
    axes[0].set_title('Rubric Score MAE by Dimension')
    axes[0].tick_params(axis='x', rotation=30)
    axes[0].axhline(y=1.0, color='green', linestyle='--', alpha=0.5, label='Good (<1.0)')
    axes[0].axhline(y=2.0, color='orange', linestyle='--', alpha=0.5, label='Acceptable (<2.0)')
    axes[0].legend(fontsize=8)
    for bar, val in zip(bars, dim_maes):
        axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.05,
                      f'{val:.2f}', ha='center', fontsize=9)

# 2. Issue Detection F1 distribution
if issue_id_matches:
    f1_values = [m['f1'] for m in issue_id_matches]
    axes[1].hist(f1_values, bins=10, edgecolor='black', alpha=0.7, color='#4C72B0')
    axes[1].axvline(np.mean(f1_values), color='red', linestyle='--',
                     label=f'Mean F1={np.mean(f1_values):.2f}')
    axes[1].set_xlabel('F1 Score')
    axes[1].set_ylabel('Count')
    axes[1].set_title('Issue Detection F1 Distribution')
    axes[1].legend()

# 3. Predicted vs Reference rubric scores scatter
pred_all_scores = []
ref_all_scores = []
for pred in predictions:
    if pred['generated_parsed'] and pred['reference']:
        p_rub = pred['generated_parsed'].get('rubricscores', {})
        r_rub = pred['reference'].get('rubricscores', {})
        for dim in RUBRIC_DIMS:
            ps = p_rub.get(dim)
            rs = r_rub.get(dim)
            if isinstance(ps, (int, float)) and isinstance(rs, (int, float)):
                pred_all_scores.append(ps)
                ref_all_scores.append(rs)

if pred_all_scores:
    axes[2].scatter(ref_all_scores, pred_all_scores, alpha=0.3, s=20, color='#8172B3')
    axes[2].plot([0, 10], [0, 10], 'r--', alpha=0.5, label='Perfect agreement')
    axes[2].set_xlabel('Reference Score')
    axes[2].set_ylabel('Predicted Score')
    axes[2].set_title('Predicted vs Reference Rubric Scores')
    axes[2].set_xlim(-0.5, 10.5)
    axes[2].set_ylim(-0.5, 10.5)
    axes[2].legend()
    # Add correlation
    if len(pred_all_scores) > 2:
        from scipy import stats
        r, p = stats.pearsonr(ref_all_scores, pred_all_scores)
        axes[2].annotate(f'r={r:.3f}', xy=(0.05, 0.92), xycoords='axes fraction', fontsize=11)

plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'semantic_evaluation.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved: semantic_evaluation.png")

In [ ]:
# ═══════════════════════════════════════════════════════════════
# COMPREHENSIVE RESULTS SUMMARY TABLE
# ═══════════════════════════════════════════════════════════════

summary_data = []

# Training metrics
summary_data.append(("Training", "Final train loss", f"{train_result.training_loss:.4f}"))
summary_data.append(("Training", "Validation loss", f"{eval_results.get('eval_loss', 'N/A'):.4f}"
                      if isinstance(eval_results.get('eval_loss'), float) else "N/A"))
summary_data.append(("Training", "Validation perplexity", f"{perplexity:.2f}"))
summary_data.append(("Training", "Training duration", f"{elapsed/60:.1f} min"))
summary_data.append(("Training", "Trainable parameters", f"{trainable_params:,} ({trainable_pct:.2f}%)"))

# Structural metrics
for metric, value in structural_metrics.items():
    summary_data.append(("Structural", metric, f"{value:.1%}"))

# Semantic metrics
if overall_mae_values:
    summary_data.append(("Semantic", "Overall Rubric MAE", f"{np.mean(overall_mae_values):.2f}"))
if issue_id_matches:
    summary_data.append(("Semantic", "Issue Detection F1", f"{avg_f1:.2%}"))
if blocking_issue_matches:
    summary_data.append(("Semantic", "Blocking Agreement", f"{agreement_rate:.2%}"))

summary_df = pd.DataFrame(summary_data, columns=["Category", "Metric", "Value"])
print(f"\n{'='*60}")
print(f"COMPREHENSIVE RESULTS SUMMARY")
print(f"{'='*60}")
print(summary_df.to_string(index=False))

summary_df.to_csv(OUTPUT_DIR / 'results_summary.csv', index=False)
print(f"\nSummary saved to {OUTPUT_DIR / 'results_summary.csv'}")

---
## Section 11 — Model Export

We save the model in two formats:

1. **LoRA Adapter Only** (~50-100MB): Just the trained adapter weights. Requires loading the base
   Phi-3 model and applying the adapter at inference time. Most storage-efficient.

2. **Merged Model** (~7.5GB in FP16): The adapter weights merged back into the base model.
   Ready for direct inference without PEFT. Required for deployment or further fine-tuning.

In [ ]:
# ═══════════════════════════════════════════════════════════════
# SAVE LoRA ADAPTER
# ═══════════════════════════════════════════════════════════════

adapter_dir = FINAL_MODEL_DIR / "lora_adapter"
adapter_dir.mkdir(exist_ok=True)

# Save the adapter
model.save_pretrained(str(adapter_dir))
tokenizer.save_pretrained(str(adapter_dir))

print(f"LoRA adapter saved to: {adapter_dir}")

# Calculate adapter size
adapter_size = sum(
    f.stat().st_size for f in adapter_dir.rglob('*') if f.is_file()
) / 1e6
print(f"Adapter size: {adapter_size:.1f} MB")

In [ ]:
# ═══════════════════════════════════════════════════════════════
# SAVE MERGED MODEL (Optional — requires more disk space)
# ═══════════════════════════════════════════════════════════════

SAVE_MERGED = True  # Set to False if disk space is limited

if SAVE_MERGED:
    print("Merging adapter with base model...")
    merged_dir = FINAL_MODEL_DIR / "merged_model"
    merged_dir.mkdir(exist_ok=True)

    # Merge and unload LoRA weights
    merged_model = model.merge_and_unload()

    # Save in FP16
    merged_model.save_pretrained(
        str(merged_dir),
        safe_serialization=True,
        max_shard_size="2GB",
    )
    tokenizer.save_pretrained(str(merged_dir))

    merged_size = sum(
        f.stat().st_size for f in merged_dir.rglob('*') if f.is_file()
    ) / 1e9
    print(f"Merged model saved to: {merged_dir}")
    print(f"Merged model size: {merged_size:.2f} GB")
else:
    print("Skipping merged model save (SAVE_MERGED=False)")

In [ ]:
# ═══════════════════════════════════════════════════════════════
# SAVE TRAINING CONFIGURATION FOR REPRODUCIBILITY
# ═══════════════════════════════════════════════════════════════

training_record = {
    "model_name": MODEL_NAME,
    "dataset_path": DATASET_PATH,
    "dataset_size": len(raw_data),
    "train_size": len(train_dataset),
    "val_size": len(val_dataset),
    "config": {k: str(v) if not isinstance(v, (int, float, bool, str, list)) else v
               for k, v in CONFIG.items()},
    "lora_config": {
        "r": CONFIG["lora_r"],
        "alpha": CONFIG["lora_alpha"],
        "dropout": CONFIG["lora_dropout"],
        "target_modules": CONFIG["target_modules"],
    },
    "trainable_parameters": trainable_params,
    "trainable_percentage": trainable_pct,
    "training_loss": train_result.training_loss,
    "eval_loss": eval_results.get('eval_loss'),
    "perplexity": perplexity,
    "structural_metrics": structural_metrics,
    "semantic_metrics": semantic_metrics,
    "training_duration_minutes": elapsed / 60,
    "seed": SEED,
}

with open(FINAL_MODEL_DIR / 'training_record.json', 'w') as f:
    json.dump(training_record, f, indent=2, default=str)

print(f"Training record saved to: {FINAL_MODEL_DIR / 'training_record.json'}")
print(f"\n{'='*60}")
print(f"ALL OUTPUTS")
print(f"{'='*60}")
print(f"  LoRA adapter:     {adapter_dir}")
if SAVE_MERGED:
    print(f"  Merged model:     {merged_dir}")
print(f"  Training curves:  {OUTPUT_DIR / 'training_results.png'}")
print(f"  Semantic eval:    {OUTPUT_DIR / 'semantic_evaluation.png'}")
print(f"  Results summary:  {OUTPUT_DIR / 'results_summary.csv'}")
print(f"  Training record:  {FINAL_MODEL_DIR / 'training_record.json'}")
print(f"  TensorBoard logs: {LOGS_DIR}")
print(f"\nTo view TensorBoard: tensorboard --logdir {LOGS_DIR}")

---
## Summary & Key Results

This notebook fine-tuned **Phi-3-mini-4k-instruct** using **QLoRA** to serve as the AuditorAgent
in an architectural governance multi-agent system.

### Training Approach
- **Method**: QLoRA (4-bit NormalFloat quantisation + Low-Rank Adaptation)
- **Trainable parameters**: ~0.2% of total model parameters
- **Data**: ~815 validated, cleaned synthetic training rows
- **Split**: Chain-aware 85/15 train/validation split (no data leakage)
- **Epochs**: 3 with cosine learning rate schedule

### Evaluation Framework
Two complementary evaluation layers:
1. **Structural metrics**: JSON validity, schema compliance, rubric range, issue template format, enum validity
2. **Semantic metrics**: Rubric score MAE, issue detection F1, blocking issue agreement

### How to Use the Model

```python
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel

# Option 1: Load adapter
base_model = AutoModelForCausalLM.from_pretrained("microsoft/Phi-3-mini-4k-instruct")
model = PeftModel.from_pretrained(base_model, "training_output/auditor_agent_model/lora_adapter")

# Option 2: Load merged model
model = AutoModelForCausalLM.from_pretrained("training_output/auditor_agent_model/merged_model")
```

### Limitations & Future Work
- **Dataset size**: 815 rows is sufficient for domain specialisation but additional data would improve generalisation
- **Rubric correlation**: High correlation between some rubric dimensions inherited from training data
- **Context length**: 4K context limits handling of very large architecture plans
- **Evaluation**: Structural metrics are deterministic; semantic quality would benefit from human evaluation